# Business Understanding and Analytic Approach
# Nhóm 4 — Dự báo nhu cầu sạc xe điện theo giờ tại thành phố Palo Alto

## 1. Giới thiệu bài toán

Sự phổ biến ngày càng tăng của xe điện (EV) tại các đô thị như Palo Alto, California đặt ra một thách thức vận hành quan trọng: hạ tầng sạc công cộng cần được quy hoạch và điều phối dựa trên nhu cầu thực tế, thay vì phỏng đoán cảm tính. Nhu cầu sạc biến động không đều theo giờ, theo trạm, và chịu ảnh hưởng của các yếu tố ngoại cảnh như thời tiết — nếu không dự báo được, thành phố dễ rơi vào tình trạng quá tải cục bộ (một số trạm bị nghẽn giờ cao điểm) hoặc lãng phí đầu tư (mở rộng trạm ở nơi ít nhu cầu).

Dự án này khai thác dữ liệu mở "Electric Vehicle Charging Station Usage" (07/2011–12/2020) của thành phố Palo Alto, kết hợp dữ liệu thời tiết theo giờ, nhằm:

- Định vị bức tranh nhu cầu: xác định quy luật biến động theo giờ, theo trạm, và vai trò của yếu tố thời tiết.
- Định lượng hiệu năng dự báo: so sánh nhiều thuật toán học máy để chọn ra mô hình dự báo nhu cầu sạc theo giờ chính xác nhất.
- Đánh giá độ tin cậy trong vận hành thực tế: kiểm tra mô hình có còn chính xác khi hoàn cảnh thay đổi (regime shift), khi áp dụng cho trạm mới, và có thể dùng để cảnh báo sớm giờ cao điểm hay không.

Đề tài gắn liền với Mục tiêu Phát triển Bền vững của Liên Hợp Quốc: **SDG 7** (năng lượng sạch, giá hợp lý) và **SDG 11** (đô thị và cộng đồng bền vững).

## 2. Mục tiêu phân tích

Dự án tập trung giải quyết 6 câu hỏi nghiên cứu khả thi (đã có bằng chứng số liệu ban đầu từ giai đoạn khám phá dữ liệu — EDA), nhằm định lượng hoá nhu cầu sạc xe điện tại Palo Alto:

- **RQ1 (Chu kỳ):** Nhu cầu sạc xe điện biến động theo chu kỳ thời gian (giờ trong ngày, thứ trong tuần, tháng trong năm) như thế nào?
- **RQ2 (Khác biệt giữa trạm):** Mức độ nhu cầu và tỷ lệ giờ cao điểm khác nhau ra sao giữa các trạm sạc trong thành phố?
- **RQ3 (Môi trường):** Nhiệt độ và lượng mưa ảnh hưởng đến nhu cầu sạc ở mức độ nào, và mối quan hệ này có tuyến tính không?
- **RQ4 (Mô hình):** Trong số nhiều thuật toán học máy (tuyến tính đến cây quyết định tập hợp), thuật toán nào cho hiệu năng dự báo tốt nhất so với mốc so sánh đơn giản (naive baseline)?
- **RQ5 (Độ bền vận hành):** Mô hình dự báo có còn chính xác khi điều kiện thực tế thay đổi (ví dụ cú sốc COVID-19) hoặc khi áp dụng cho trạm sạc chưa từng xuất hiện trong dữ liệu huấn luyện?
- **RQ6 (Ứng dụng):** Làm thế nào để chuyển kết quả dự báo thành cảnh báo sớm giờ cao điểm, hỗ trợ đơn vị vận hành lưới điện và trạm sạc ra quyết định kịp thời?

## 3. Các yếu tố ảnh hưởng đến nhu cầu sạc xe điện

Dựa trên dữ liệu đã thu thập, các yếu tố chính bao gồm:

- **Thời gian (ts):** quyết định tính chu kỳ của nhu cầu sạc (giờ cao điểm vs. thấp điểm; ngày thường vs. cuối tuần).
- **Lịch sử gần đây (lag features):** nhu cầu ở giờ liền trước có tương quan rất mạnh với nhu cầu hiện tại (autocorrelation lag-1 ≈ 0,77).
- **Trạm sạc (station):** mỗi trạm có mức nhu cầu nền và tỷ lệ giờ cao điểm khác nhau, do vị trí và loại hình sử dụng khác nhau (văn phòng, thư viện, khu dân cư...).
- **Thời tiết (temp_c, precip_mm):** nhiệt độ có quan hệ phi tuyến với nhu cầu (trời lạnh hơn có xu hướng làm tăng nhẹ nhu cầu sạc); lượng mưa ảnh hưởng không đáng kể.

## 4. Cách tiếp cận phân tích

Để đạt được mục tiêu trên, dự án thực hiện các bước sau:

1. **Thu thập và làm sạch dữ liệu:**
   - Đăng ký minh bạch nguồn dữ liệu chính (Palo Alto Open Data Portal) và nguồn phụ (dữ liệu thời tiết theo giờ, khu vực Moffett Field).
   - Loại bỏ bản ghi trùng lặp và giá trị ngoại lai (0,1% cực trị hai đầu) theo nhật ký làm sạch có ghi lý do (cleaning log).
   - Đưa dữ liệu về lưới thời gian đều theo giờ, nội suy khoảng trống ngắn (≤ 3 giờ), loại các trạm không đủ lịch sử.
   - Ghép nguồn thời tiết vào bảng chính theo thời điểm gần nhất.

2. **Phân tích SQL:**
   - Truy vấn tính trung bình và độ lệch nhu cầu theo trạm và theo tháng.
   - Truy vấn tỷ lệ giờ vượt phân vị 90% (giờ cao điểm) theo từng trạm.
   - Truy vấn mối quan hệ giữa nhu cầu và phân vị nhiệt độ (decile).
   - Kiểm định thống kê chính thức (Kruskal-Wallis giữa các trạm; tương quan Spearman với biến thời tiết).

3. **Phân tích khám phá dữ liệu (EDA) bằng Python:**
   - Kiểm tra độ lệch phân phối của biến mục tiêu (skewness), cân nhắc biến đổi log khi mô hình hoá.
   - Đo độ mạnh mùa vụ theo giờ / thứ / tháng.
   - Phân tích tự tương quan (ACF) để xác định độ trễ (lag) phù hợp làm đặc trưng.
   - Phân tích tương quan chéo có độ trễ giữa thời tiết và nhu cầu.

4. **Trực quan hoá dữ liệu:**
   - Biểu đồ chuỗi thời gian theo trạm, đánh dấu mốc chia train/test.
   - Biểu đồ mùa vụ theo giờ / thứ / tháng.
   - Bản đồ nhiệt tỷ lệ dữ liệu thiếu theo tháng và theo trạm.

5. **Xây dựng mô hình dự báo:**
   - Tạo đặc trưng: giá trị trễ (1, 2, 3, 24, 48, 168 giờ), trung bình/độ lệch chuẩn trượt theo mùa, đặc trưng lịch, đặc trưng thời tiết, định danh trạm.
   - Kiểm tra rò rỉ dữ liệu (leakage check) trước khi huấn luyện.
   - Huấn luyện và so sánh nhiều họ mô hình (hồi quy tuyến tính/regularized đến cây quyết định tập hợp: Random Forest, LightGBM, XGBoost...), đo bằng MAE, RMSE, MASE, so với mốc naive.
   - Kiểm tra độ bền: đánh giá theo phương pháp lùi gốc dự báo (rolling-origin) qua nhiều mốc thời gian, và trên các trạm chưa từng xuất hiện trong huấn luyện (unseen station).
   - Ước lượng khoảng tin cậy dự báo (conformal prediction) và xây bộ phân loại cảnh báo giờ cao điểm.
   - Giải thích mô hình bằng SHAP và nghiên cứu loại bỏ đặc trưng (ablation study).

6. **Khuyến nghị vận hành và giá trị thực tiễn:**
   - Trích xuất danh sách trạm có tỷ lệ giờ cao điểm cao nhất, cần ưu tiên giám sát hoặc mở rộng công suất.
   - Đề xuất ngưỡng cảnh báo sớm dựa trên chi phí (cost-sensitive threshold) giữa bỏ sót cảnh báo và cảnh báo giả.
   - Đề xuất khung thời gian huấn luyện lại mô hình định kỳ để thích ứng với thay đổi hành vi sử dụng theo thời gian.

## 5. Kỳ vọng kết quả

- **Tri thức định lượng:** làm rõ quy luật chu kỳ, sự khác biệt giữa các trạm, và vai trò (hạn chế) của thời tiết đối với nhu cầu sạc xe điện tại Palo Alto.
- **Giải pháp hỗ trợ quyết định:** mô hình dự báo có độ chính xác cao hơn mốc naive rõ rệt, kèm khoảng tin cậy và cảnh báo sớm giờ cao điểm, hỗ trợ đơn vị vận hành lưới điện và trạm sạc.
- **Sản phẩm học thuật:** bộ notebook Jupyter tái lập được (reproducible), nhật ký minh bạch việc dùng AI (AI Audit Log), và báo cáo nghiên cứu dạng hội nghị đáp ứng chuẩn mực học thuật.
